# GRIDPILOT AI — Member 2 — Task 7
## Demand Spike Classifier (XGBoost)

**Purpose:** Define a reproducible, distribution-based labeling rule for Normal/Moderate/Severe demand spikes, then train and evaluate an XGBoost classifier against it.


## 0. Setup

In [ ]:
!pip -q install pandas numpy scikit-learn xgboost pyarrow

import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix, classification_report
import xgboost as xgb

print("Libraries loaded. XGBoost version:", xgb.__version__)

## 1. Load feature-engineered dataset

In [ ]:
FEATURES_PATH = "./feature_engineering_outputs/demand_features.parquet"

df = pd.read_parquet(FEATURES_PATH)
df = df.sort_values("timestamp").reset_index(drop=True)
DEMAND_COL = "demand" if "demand" in df.columns else df.columns[1]
print(df.shape)

## 2. Define a reproducible, documented spike-labeling rule

Classes:
- 0 = Normal
- 1 = Moderate Spike
- 2 = Severe Spike

The thresholds below are derived from the historical demand-growth distribution -- not hardcoded guesses. Adjust the percentile cut points and document why you chose them.

In [ ]:
FREQ_MINUTES = None  # set from Task 1 / Chunk 3
assert FREQ_MINUTES is not None, "Set FREQ_MINUTES before running."

df["historical_peak"] = df[DEMAND_COL].cummax()
df["load_growth_pct"] = df[DEMAND_COL].pct_change() * 100

# Derive thresholds from the distribution of positive growth values (documented, not arbitrary)
positive_growth = df.loc[df["load_growth_pct"] > 0, "load_growth_pct"]
MODERATE_THRESHOLD = positive_growth.quantile(0.90)
SEVERE_THRESHOLD = positive_growth.quantile(0.99)

print(f"Moderate spike threshold (90th pct of positive growth): {MODERATE_THRESHOLD:.3f}%")
print(f"Severe spike threshold (99th pct of positive growth): {SEVERE_THRESHOLD:.3f}%")

def label_spike(growth_pct):
    if pd.isna(growth_pct):
        return np.nan
    if growth_pct >= SEVERE_THRESHOLD:
        return 2
    if growth_pct >= MODERATE_THRESHOLD:
        return 1
    return 0

df["spike_class"] = df["load_growth_pct"].apply(label_spike)
df["spike_class"].value_counts(dropna=False)

## 3. Feature set for the classifier

In [ ]:
WEATHER_FEATURE_CANDIDATES = ["temperature", "humidity", "cloud_cover", "wind_speed", "solar_radiation"]
present_weather = [c for c in WEATHER_FEATURE_CANDIDATES if c in df.columns]

CLASSIFIER_FEATURES = [
    c for c in [
        DEMAND_COL,
        f"{DEMAND_COL}_lag_15min",
        "historical_peak",
        "load_growth_pct",
        "hour",
        "day_of_week",
    ] + present_weather
    if c in df.columns
]
print("Classifier features:", CLASSIFIER_FEATURES)

## 4. Chronological split

In [ ]:
valid_df = df.dropna(subset=CLASSIFIER_FEATURES + ["spike_class"]).copy()

n = len(valid_df)
train_end = int(n * 0.7)
val_end = int(n * 0.85)

train_df = valid_df.iloc[:train_end]
val_df = valid_df.iloc[train_end:val_end]
test_df = valid_df.iloc[val_end:]

print(len(train_df), len(val_df), len(test_df))

## 5. Train the XGBoost classifier

In [ ]:
XGB_PARAMS = {
    "objective": "multi:softprob",
    "num_class": 3,
    "eval_metric": "mlogloss",
    "eta": 0.05,
    "max_depth": 5,
    "subsample": 0.9,
    "colsample_bytree": 0.9,
    "seed": 42,
}
NUM_BOOST_ROUND = 300
EARLY_STOPPING_ROUNDS = 20

dtrain = xgb.DMatrix(train_df[CLASSIFIER_FEATURES], label=train_df["spike_class"])
dval = xgb.DMatrix(val_df[CLASSIFIER_FEATURES], label=val_df["spike_class"])
dtest = xgb.DMatrix(test_df[CLASSIFIER_FEATURES], label=test_df["spike_class"])

spike_model = xgb.train(
    XGB_PARAMS,
    dtrain,
    num_boost_round=NUM_BOOST_ROUND,
    evals=[(dval, "validation")],
    early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    verbose_eval=False,
)
print("Best iteration:", spike_model.best_iteration)

## 6. Evaluate: precision, recall, F1, confusion matrix

In [ ]:
pred_probs = spike_model.predict(dtest)
pred_classes = np.argmax(pred_probs, axis=1)
true_classes = test_df["spike_class"].astype(int).values

precision, recall, f1, support = precision_recall_fscore_support(
    true_classes, pred_classes, labels=[0, 1, 2], zero_division=0
)

metrics_df = pd.DataFrame({
    "class": ["Normal", "Moderate Spike", "Severe Spike"],
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "support": support,
})
display(metrics_df)

cm = confusion_matrix(true_classes, pred_classes, labels=[0, 1, 2])
cm_df = pd.DataFrame(cm, index=["true_0", "true_1", "true_2"], columns=["pred_0", "pred_1", "pred_2"])
display(cm_df)

print(classification_report(true_classes, pred_classes, target_names=["Normal", "Moderate", "Severe"]))

## 7. Save model, thresholds, and evaluation artifacts

In [ ]:
MODEL_VERSION = "spike-xgb-v1"
OUTPUT_DIR = Path("./ml_models_spike")
OUTPUT_DIR.mkdir(exist_ok=True)

spike_model.save_model(str(OUTPUT_DIR / f"{MODEL_VERSION}.json"))

labeling_rule = {
    "model_version": MODEL_VERSION,
    "moderate_threshold_pct": float(MODERATE_THRESHOLD),
    "severe_threshold_pct": float(SEVERE_THRESHOLD),
    "derivation": "90th/99th percentile of positive load_growth_pct in training history",
    "features": CLASSIFIER_FEATURES,
}
with open(OUTPUT_DIR / f"{MODEL_VERSION}_labeling_rule.json", "w") as f:
    json.dump(labeling_rule, f, indent=2)

metrics_df.to_csv(OUTPUT_DIR / f"{MODEL_VERSION}_test_metrics.csv", index=False)
cm_df.to_csv(OUTPUT_DIR / f"{MODEL_VERSION}_confusion_matrix.csv")

print("Saved to:", OUTPUT_DIR)
for p in sorted(OUTPUT_DIR.iterdir()):
    print("-", p.name)

## Next step

The labeling thresholds are documented in `spike-xgb-v1_labeling_rule.json` -- do not hardcode a different threshold in `services/forecasting/**` without updating this file. Port the trained classifier into the forecast service, then move on to Chunk 8 (tests).